In [1]:
# Part 4:
# Setup
!pip install crewai crewai-tools -q
!pip install litellm -q
!pip install nest_asyncio -q

import nest_asyncio
nest_asyncio.apply()

from google.colab import userdata
import os

os.environ['GROQ_API_KEY'] = userdata.get('GROQ_API_KEY')

# Fix for a known CrewAI bug: disable an internal caching feature that Groq doesn't support
import crewai.llms.cache as _crewai_cache
_crewai_cache.mark_cache_breakpoint = lambda msg: msg

print("Setup complete")

Setup complete


In [2]:
# PART 4 — Task 1 & 2
# SAFE LIVE API TOOLS

from crewai.tools import tool
import requests

# Tool 1 — Random Joke

@tool("Get Random Joke")
def get_random_joke(topic: str = "any") -> str:
    """
    Fetch a random joke from the live Official Joke API.

    Use this tool when the user asks for a joke, something funny,
    or a light-hearted message.

    Parameters:
        topic: Optional topic supplied by the agent.
                The current API endpoint returns a random joke.

    Returns:
        The joke text, or a safe error message if the API fails.
    """

    try:
        response = requests.get(
            "https://official-joke-api.appspot.com/random_joke",
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        if not isinstance(data, dict):
            return "Tool error: joke API returned an unexpected response."

        if "setup" not in data or "punchline" not in data:
            return "Tool error: joke API response was missing required fields."

        return f"{data['setup']} — {data['punchline']}"

    except requests.RequestException as e:
        return f"Tool error: joke API request failed safely ({type(e).__name__})."

    except (ValueError, KeyError, TypeError):
        return "Tool error: joke API returned invalid data."

    except Exception as e:
        return f"Tool error: unexpected joke tool failure ({type(e).__name__})."


# Tool 2 — Life Advice

@tool("Get Life Advice")
def get_advice(topic: str = "any") -> str:
    """
    Fetch a random piece of life advice from the live Advice Slip API.

    Use this tool when the user asks for life advice,
    motivation, or guidance.

    Parameters:
        topic: Optional topic supplied by the agent.
                The current API endpoint returns random advice.

    Returns:
        The advice text, or a safe error message if the API fails.
    """

    try:
        response = requests.get(
            "https://api.adviceslip.com/advice",
            timeout=10
        )

        response.raise_for_status()

        data = response.json()

        if not isinstance(data, dict):
            return "Tool error: advice API returned an unexpected response."

        if "slip" not in data or not isinstance(data["slip"], dict):
            return "Tool error: advice API response was missing the slip object."

        if "advice" not in data["slip"]:
            return "Tool error: advice API response was missing the advice field."

        return data["slip"]["advice"]

    except requests.RequestException as e:
        return f"Tool error: advice API request failed safely ({type(e).__name__})."

    except (ValueError, KeyError, TypeError):
        return "Tool error: advice API returned invalid data."

    except Exception as e:
        return f"Tool error: unexpected advice tool failure ({type(e).__name__})."


print("Both safe live-API tools are ready.")

Both safe live-API tools are ready.


In [3]:
# STEP 2 — Native CrewAI tool-call logger
# Uses CrewAI's native event system to capture actual tool calls
# and their parsed arguments.

from pprint import pprint

from crewai.events import (
    BaseEventListener,
    ToolUsageStartedEvent,
    ToolUsageFinishedEvent,
    ToolUsageErrorEvent,
    ToolExecutionErrorEvent,
    ToolSelectionErrorEvent,
    ToolValidateInputErrorEvent,
)


class ToolCallLogger(BaseEventListener):
    """
    Native CrewAI event listener for inspecting tool calls.

    Captures the tool name and parsed arguments directly from
    CrewAI's native tool-calling event system.
    """

    def __init__(self):
        super().__init__()
        print("Native CrewAI tool-call logger initialized.")

    def setup_listeners(self, crewai_event_bus):

        @crewai_event_bus.on(ToolUsageStartedEvent)
        def on_tool_started(source, event):
            print("\n======================================")
            print("NATIVE CREWAI TOOL CALL")
            print("======================================")

            tool_name = getattr(event, "tool_name", None)
            tool_args = getattr(event, "tool_args", None)

            tool_call = {
                "tool": tool_name,
                "arguments": tool_args
            }

            print("Structured tool-call object:")
            pprint(tool_call)

        @crewai_event_bus.on(ToolUsageFinishedEvent)
        def on_tool_finished(source, event):
            print("\nTool finished:")
            print("Tool:", getattr(event, "tool_name", None))

        @crewai_event_bus.on(ToolUsageErrorEvent)
        def on_tool_error(source, event):
            print("\nTool usage error:")
            pprint(event.__dict__)

        @crewai_event_bus.on(ToolExecutionErrorEvent)
        def on_execution_error(source, event):
            print("\nTool execution error:")
            pprint(event.__dict__)

        @crewai_event_bus.on(ToolSelectionErrorEvent)
        def on_selection_error(source, event):
            print("\nTool selection error:")
            pprint(event.__dict__)

        @crewai_event_bus.on(ToolValidateInputErrorEvent)
        def on_validation_error(source, event):
            print("\nTool validation error:")
            pprint(event.__dict__)


# Create the listener instance.
# This registers the listeners with CrewAI's event bus.
tool_call_logger = ToolCallLogger()

print("Native CrewAI tool-call logging is ready.")

Native CrewAI tool-call logger initialized.
Native CrewAI tool-call logging is ready.


In [4]:
# Setting up the AI model that will power our agents
from crewai import Agent, Task, Crew, Process, LLM

my_llm = LLM(model="groq/llama-3.3-70b-versatile", temperature=0.3)

print("The LLM is set up and ready to use")

The LLM is set up and ready to use


In [5]:
# TAsk :
# Creating two agents, each with their own role
from crewai import Agent

# Agent 1: this agent's job is to fetch content using our tools
fetcher_agent = Agent(
    role="Content Fetcher",
    goal="Fetch a joke and a piece of life advice using the available tools",
    backstory="You are a cheerful assistant who loves finding fun and useful content "
               "for people. You always use your tools to get real, fresh content "
               "instead of making things up.",
    tools=[get_random_joke, get_advice],
    llm=my_llm,
    allow_delegation=False,
    verbose=True
)

# Agent 2: this agent's job is to take the fetched content and write a nice message
writer_agent = Agent(
    role="Message Writer",
    goal="Take the joke and advice that were fetched, and turn them into a warm, friendly message",
    backstory="You are a skilled writer who takes raw pieces of content and turns them "
               "into something pleasant and easy to read for the end user.",
    tools=[],
    llm=my_llm,
    allow_delegation=True,
    verbose=True
)

print("Both agents are created")

Both agents are created


In [6]:
# Task 2 :
# Creating two tasks - one for each agent
from crewai import Task

# Task 1: the fetcher agent's job
fetch_task = Task(
    description="Fetch one joke and one piece of life advice using your tools.",
    expected_output="A joke and a piece of advice, clearly labeled.",
    agent=fetcher_agent
)

# Task 2: the writer agent's job - this uses Task 1's result as context
write_task = Task(
    description="Using the joke and advice that were fetched, write one short, "
                 "warm, friendly paragraph combining both into a nice message for the user.",
    expected_output="A short friendly paragraph combining the joke and the advice.",
    agent=writer_agent,
    context=[fetch_task]
)

print("Both tasks are created")

Both tasks are created


In [7]:
# Task 3 :
# a :
# Assemble the crew and run it - SEQUENTIAL process
from crewai import Crew, Process

my_crew = Crew(
    agents=[fetcher_agent, writer_agent],
    tasks=[fetch_task, write_task],
    process=Process.sequential,
    verbose=True
)

result = await my_crew.kickoff_async()

print("\n\nFINAL RESULT:")
print(result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 4b0c6c08-2bc8-4c50-a905-ed2b490fee35                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Fetch one joke and one piece of life advice using your tools.                                            │
│  ID: 7b8e8059-b242-47a4-9b25-b05818321179                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: Fetch one joke and one piece of life advice using your tools.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


NATIVE CREWAI TOOL CALL
Structured tool-call object:
{'arguments': {'topic': 'any'}, 'tool': 'get_random_joke'}


╭──────────────────────────────────────── 🔧 Tool Execution Started (#1) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_random_joke                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Message Writer                                                                                          │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I've got a joke and some advice for you. Why do C# and Java developers keep breaking their keyboards? —        │
│  Because they use a strongly typed language. On a more serious note, sometimes it's best to ignore other        │
│  people's advice.                                                                                               │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the joke and advice that were fetched, write one short, warm, friendly paragraph combining both    │
│  into a nice message for the user.                                                                              │
│  Agent: Message Writer                                                                                          │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



FINAL RESULT:
I've got a joke and some advice for you. Why do C# and Java developers keep breaking their keyboards? — Because they use a strongly typed language. On a more serious note, sometimes it's best to ignore other people's advice.


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 4b0c6c08-2bc8-4c50-a905-ed2b490fee35                                                                       │
│  Final Output: I've got a joke and some advice for you. Why do C# and Java developers keep breaking their       │
│  keyboards? — Because they use a strongly typed language. On a more serious note, sometimes it's best to        │
│  ignore other people's advice.                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [8]:
# b :
# Run the crew again - HIERARCHICAL process (with a manager agent deciding order)
my_crew_hierarchical = Crew(
    agents=[fetcher_agent, writer_agent],
    tasks=[fetch_task, write_task],
    process=Process.hierarchical,
    manager_llm=my_llm,
    verbose=True
)

result_hierarchical = await my_crew_hierarchical.kickoff_async()

print("\n\nFINAL RESULT (hierarchical):")
print(result_hierarchical)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6f0ac6c7-87d7-4749-9954-9abf9dfc9ecf                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Fetch one joke and one piece of life advice using your tools.                                            │
│  ID: 7b8e8059-b242-47a4-9b25-b05818321179                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Task: Fetch one joke and one piece of life advice using your tools.                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


NATIVE CREWAI TOOL CALL
Structured tool-call object:
{'arguments': {'topic': 'any'}, 'tool': 'get_random_joke'}

NATIVE CREWAI TOOL CALL
Structured tool-call object:
{'arguments': {'topic': 'any'}, 'tool': 'get_life_advice'}


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I hope you're having a fantastic day. I just heard a joke that made me think of you, and I had to share it -   │
│  what do you call a dog that can do magic tricks? A labracadabrador, isn't that just the best? On a more        │
│  practical note, I also wanted to pass on a tip that I recently learned: Never pay full price for a sofa at     │
│  DFS. It's amazing how much you can save with a little patience and the right discount. So, whether you're in   │
│  the market for a new sofa or just need a smile, I hope this helps brighten your day and reminds you to always  │
│  keep an eye out for those magical deals!                                                                       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Using the joke and advice that were fetched, write one short, warm, friendly paragraph combining both    │
│  into a nice message for the user.                                                                              │
│  Agent: Crew Manager                                                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



FINAL RESULT (hierarchical):
I hope you're having a fantastic day. I just heard a joke that made me think of you, and I had to share it - what do you call a dog that can do magic tricks? A labracadabrador, isn't that just the best? On a more practical note, I also wanted to pass on a tip that I recently learned: Never pay full price for a sofa at DFS. It's amazing how much you can save with a little patience and the right discount. So, whether you're in the market for a new sofa or just need a smile, I hope this helps brighten your day and reminds you to always keep an eye out for those magical deals!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6f0ac6c7-87d7-4749-9954-9abf9dfc9ecf                                                                       │
│  Final Output: I hope you're having a fantastic day. I just heard a joke that made me think of you, and I had   │
│  to share it - what do you call a dog that can do magic tricks? A labracadabrador, isn't that just the best?    │
│  On a more practical note, I also wanted to pass on a tip that I recently learned: Never pay full price for a   │
│  sofa at DFS. It's amazing how much you can save with a little patience and the right discount. So, whether     │
│  you're in the market for a new sofa or just need a smile, I hope this helps brighten your day and reminds you  │
│  to always keep an eye out for those magical deals!                                                             │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [9]:
# COMMON REQUIREMENT 4: Demonstrate on 3 distinct queries (query 2 of 3)
from crewai import Task, Crew, Process

query2_task = Task(
    description="Get me a joke to brighten my day.",
    expected_output="A single joke.",
    agent=fetcher_agent
)

query2_crew = Crew(agents=[fetcher_agent], tasks=[query2_task], process=Process.sequential, verbose=True)
query2_result = await query2_crew.kickoff_async()

print("\n\nQUERY 2 RESULT:")
print(query2_result)

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6a879c10-0a7a-4710-95f3-92d596a02eed                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: Get me a joke to brighten my day.                                                                        │
│  ID: 52eab604-e55c-40a2-b97a-3d9dd77bd46a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: Get me a joke to brighten my day.                                                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


NATIVE CREWAI TOOL CALL
Structured tool-call object:
{'arguments': {'topic': 'any'}, 'tool': 'get_random_joke'}


...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  I hope that joke brightened your day!                                                                          │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: Get me a joke to brighten my day.                                                                        │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



QUERY 2 RESULT:
I hope that joke brightened your day!


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6a879c10-0a7a-4710-95f3-92d596a02eed                                                                       │
│  Final Output: I hope that joke brightened your day!                                                            │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [10]:
# COMMON REQUIREMENT 4: Demonstrate on 3 distinct queries (query 3 of 3)
query3_task = Task(
    description="I'm feeling stuck in life. Can you give me some advice?",
    expected_output="A single piece of life advice.",
    agent=fetcher_agent
)

query3_crew = Crew(agents=[fetcher_agent], tasks=[query3_task], process=Process.sequential, verbose=True)
query3_result = await query3_crew.kickoff_async()

print("\n\nQUERY 3 RESULT:")
print(query3_result)

╭─────────────────────────────────────────── 🚀 Crew Execution Started ───────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 6fa7faa8-8650-44dd-beb1-6e47fce25bab                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 📋 Task Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Started                                                                                                   │
│  Name: I'm feeling stuck in life. Can you give me some advice?                                                  │
│  ID: b417b740-3d1f-4555-a102-eff41a1bea1a                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Task: I'm feeling stuck in life. Can you give me some advice?                                                  │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


NATIVE CREWAI TOOL CALL
Structured tool-call object:
{'arguments': {'topic': 'any'}, 'tool': 'get_life_advice'}


╭──────────────────────────────────────── 🔧 Tool Execution Started (#6) ─────────────────────────────────────────╮
│                                                                                                                 │
│  Tool: get_life_advice                                                                                          │
│  Args: {'topic': 'any'}                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

[Finalize] todos_count=0, todos_with_results=0


╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  When you're at a concert or event, enjoy the moment, enjoy being there. Try leaving your camera in your        │
│  pocket.                                                                                                        │
│  How many lips does a flower have? — Tulips                                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────── 📋 Task Completion ───────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: I'm feeling stuck in life. Can you give me some advice?                                                  │
│  Agent: Content Fetcher                                                                                         │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯



QUERY 3 RESULT:
When you're at a concert or event, enjoy the moment, enjoy being there. Try leaving your camera in your pocket. 
How many lips does a flower have? — Tulips


╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 6fa7faa8-8650-44dd-beb1-6e47fce25bab                                                                       │
│  Final Output: When you're at a concert or event, enjoy the moment, enjoy being there. Try leaving your camera  │
│  in your pocket.                                                                                                │
│  How many lips does a flower have? — Tulips                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [11]:
# PART 4, TASK 4: Confirm delegation (allow_delegation=True)
# The Message Writer agent has allow_delegation=True.
# During the Task 3a (sequential) run, the Writer agent used the
# delegate_work_to_coworker tool to ask the Content Fetcher for a
# combined draft, since the Writer itself has no tools.
# This is a real, captured example of delegation happening mid-task
# (see the "Tool Execution Started: delegate_work_to_coworker" log
# in the Task 3a output above).

╭──────────────────────────────────────────────── Tracing Status ─────────────────────────────────────────────────╮
│                                                                                                                 │
│  Info: Tracing is disabled.                                                                                     │
│                                                                                                                 │
│  To enable tracing, do any one of these:                                                                        │
│  • Set tracing=True in your Crew/Flow code                                                                      │
│  • Set CREWAI_TRACING_ENABLED=true in your project's .env file                                                  │
│  • Run: crewai traces enable                                                                                    │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [14]:
import importlib.metadata

print("pandas:", importlib.metadata.version("pandas"))
print("crewai:", importlib.metadata.version("crewai"))
print("litellm:", importlib.metadata.version("litellm"))

pandas: 2.2.2
crewai: 1.15.14
litellm: 1.96.0


In [15]:
import importlib.metadata

print("requests:", importlib.metadata.version("requests"))

requests: 2.34.2
